In [ ]:
!pip install --no-index --find-links \\
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \\
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
# ==============================================================
# SNN-Synthesis Agent for ARC-AGI-3
# Noisy Beam Search (NBS) + Curiosity-Driven Exploration
#
# Paper: https://doi.org/10.5281/zenodo.15188587
# GitHub: https://github.com/hifunsk/snn-synthesis
# Author: Hiroto Kyan
# ==============================================================
import hashlib
import logging
import os
import random
import time
import traceback
from collections import deque
from datetime import datetime
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState

logger = logging.getLogger(__name__)

# --- All available actions ---
ALL_ACTIONS = [
    GameAction.ACTION1,
    GameAction.ACTION2,
    GameAction.ACTION3,
    GameAction.ACTION4,
]
try:
    ALL_ACTIONS.append(GameAction.ACTION5)
    ALL_ACTIONS.append(GameAction.ACTION6)
    ALL_ACTIONS.append(GameAction.ACTION7)
except AttributeError:
    pass


# ==============================================================
# State Feature Extraction
# ==============================================================
def extract_frame_features(frame: FrameData) -> list:
    """Extract numeric features from a FrameData object."""
    features = []
    # Level info
    features.append(float(frame.levels_completed or 0))
    features.append(float(frame.win_levels or 0))
    # Grid features (frame.frame is a list of 2D arrays)
    if frame.frame:
        for grid in frame.frame[:3]:  # up to 3 grids
            if isinstance(grid, (list, np.ndarray)):
                arr = np.array(grid, dtype=np.float32)
                features.append(float(arr.shape[0]) if len(arr.shape) > 0 else 0.0)
                features.append(float(arr.shape[1]) if len(arr.shape) > 1 else 0.0)
                features.append(float(arr.mean()))
                features.append(float(arr.std()))
                features.append(float(arr.max()))
                features.append(float(arr.min()))
                features.append(float(np.count_nonzero(arr)))
                # Unique values count
                features.append(float(len(np.unique(arr))))
    # Available actions
    if hasattr(frame, 'available_actions') and frame.available_actions:
        features.append(float(len(frame.available_actions)))
    # Pad or truncate to fixed size
    TARGET_DIM = 32
    if len(features) < TARGET_DIM:
        features.extend([0.0] * (TARGET_DIM - len(features)))
    return features[:TARGET_DIM]


def hash_features(features: list, precision: int = 2) -> str:
    """Hash state features for novelty tracking."""
    rounded = tuple(round(f, precision) for f in features[:16])
    return hashlib.md5(str(rounded).encode()).hexdigest()[:12]


# ==============================================================
# StateBrain: Lightweight MLP Policy
# ==============================================================
class StateBrain(nn.Module):
    def __init__(self, state_dim=32, n_actions=4, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x, noise_sigma=0.0):
        h = self.net[0](x)
        h = self.net[1](h)
        h = self.net[2](h)
        if noise_sigma > 0:
            h = h + torch.randn_like(h) * noise_sigma
        h = self.net[3](h)
        h = self.net[4](h)
        h = self.net[5](h)
        h = self.net[6](h)
        return h


# ==============================================================
# SNN-Synthesis Agent
# ==============================================================
class StochasticGoose(Agent):
    """
    SNN-Synthesis: Noisy Beam Search + Curiosity-Driven Exploration.
    Inherits from the official Agent base class.
    """
    MAX_ACTIONS = 300  # max steps per game

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # State tracking
        self.state_visit_count = {}
        self.rng = random.Random(42)
        self.step_count = 0
        self.best_levels = 0
        self.action_history = []
        self.feature_history = []

        # Noise parameters (from SNN-Synthesis v5)
        self.noise_sigma = 0.10
        self.curiosity_bonus = 3.0

        # Model (will be trained if miracles found)
        self.model = None
        self.n_actions = min(len(ALL_ACTIONS), 4)
        self.state_dim = 32

        logger.info(f"SNN-Synthesis Agent initialized for {self.game_id}")

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        """Check if agent should stop."""
        if latest_frame.state in (GameState.GAME_OVER, GameState.WIN):
            return True
        return False

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        """
        Choose action using curiosity-driven exploration with optional
        noise injection (Noisy Beam Search principle).
        """
        self.step_count += 1

        # First action: RESET
        if self.step_count == 1:
            return GameAction.RESET

        # Extract features
        try:
            features = extract_frame_features(latest_frame)
        except Exception:
            features = [0.0] * self.state_dim
        self.feature_history.append(features)

        # Track state visits for curiosity
        state_hash = hash_features(features)
        self.state_visit_count[state_hash] = (
            self.state_visit_count.get(state_hash, 0) + 1
        )

        # Track best progress
        if latest_frame.levels_completed and latest_frame.levels_completed > self.best_levels:
            self.best_levels = latest_frame.levels_completed

        # Determine available actions
        n_actions = self.n_actions
        if hasattr(latest_frame, 'available_actions') and latest_frame.available_actions:
            avail = latest_frame.available_actions
            n_actions = min(len(avail), len(ALL_ACTIONS))

        # Choose action: model-based or curiosity-random
        if self.model is not None:
            action = self._model_action(features, n_actions)
        else:
            action = self._curiosity_action(features, n_actions)

        self.action_history.append(ALL_ACTIONS.index(action))
        return action

    def _curiosity_action(self, features: list, n_actions: int) -> GameAction:
        """Choose action using curiosity bonus (UCB-style exploration)."""
        scores = []
        for a_idx in range(n_actions):
            sa_key = f"{hash_features(features)}_{a_idx}"
            visit_n = self.state_visit_count.get(sa_key, 0)
            # UCB-style: novelty bonus + small random noise
            bonus = self.curiosity_bonus / (visit_n + 1) ** 0.5
            noise = self.rng.gauss(0, 0.1)
            scores.append(bonus + noise)
            self.state_visit_count[sa_key] = visit_n

        best_idx = scores.index(max(scores))

        # Track state-action visit
        sa_key = f"{hash_features(features)}_{best_idx}"
        self.state_visit_count[sa_key] = self.state_visit_count.get(sa_key, 0) + 1

        return ALL_ACTIONS[best_idx]

    def _model_action(self, features: list, n_actions: int) -> GameAction:
        """Choose action using trained StateBrain + noise injection."""
        try:
            x = torch.tensor(features[:self.state_dim], dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                logits = self.model(x, noise_sigma=self.noise_sigma)

            # Add curiosity bonus
            for a_idx in range(min(n_actions, logits.size(1))):
                sa_key = f"{hash_features(features)}_{a_idx}"
                visit_n = self.state_visit_count.get(sa_key, 0)
                logits[0, a_idx] += self.curiosity_bonus / (visit_n + 1) ** 0.5

            probs = torch.softmax(logits[:, :n_actions] / 0.8, dim=1)
            action_idx = torch.multinomial(probs, 1).item()

            # Track visit
            sa_key = f"{hash_features(features)}_{action_idx}"
            self.state_visit_count[sa_key] = self.state_visit_count.get(sa_key, 0) + 1

            return ALL_ACTIONS[action_idx]
        except Exception:
            return self._curiosity_action(features, n_actions)


# SNN-Synthesis Agent for ARC-AGI-3

This notebook implements a **Noisy Beam Search (NBS) + Curiosity-Driven Exploration** agent.

- Paper: https://doi.org/10.5281/zenodo.15188587
- GitHub: https://github.com/hifunsk/snn-synthesis
- Author: Hiroto Kyan
